# Binary Fall Detection — FallDetectionTransformerCNNLSTM
**Classes:** 0=NO-FALL, 1=FALL | **Input:** (40,20) windows | **Output:** 2-class softmax

In [ ]:
import json,os,pickle,warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch,torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score,roc_curve
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)
print('Imports OK')

In [ ]:
MANIFEST_CSV  = '../aug_dataset_binary/dataset_manifest.csv'
WINDOW_SIZE   = 40
STRIDE        = 3
FRAME_DT      = 0.055
SNR_THRESHOLD = 10.0
BATCH_SIZE    = 16
MAX_EPOCHS    = 150
LR            = 0.001
WEIGHT_DECAY  = 0.001
PATIENCE      = 25
GRAD_CLIP     = 1.0
NUM_CLASSES   = 2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def extract_features(frame_data,prev_feat=None):
    pts=np.array(frame_data.get('pointCloud',[]))
    if len(pts)>0: pts=pts[pts[:,4]>=SNR_THRESHOLD]
    feat=np.zeros(20)
    if len(pts)>0:
        feat[0]=np.mean(pts[:,0]); feat[1]=np.mean(pts[:,1]); feat[2]=np.mean(pts[:,2])
        a=np.arctan2(pts[:,0],pts[:,1]); e=np.arctan2(pts[:,2],np.sqrt(pts[:,0]**2+pts[:,1]**2))
        feat[3]=np.mean(pts[:,3]*np.sin(a)); feat[4]=np.mean(pts[:,3]*np.cos(a)); feat[5]=np.mean(pts[:,3]*np.sin(e))
        if prev_feat is not None:
            feat[6]=(feat[3]-prev_feat[3])/FRAME_DT; feat[7]=(feat[4]-prev_feat[4])/FRAME_DT; feat[8]=(feat[5]-prev_feat[5])/FRAME_DT
        feat[9]=len(pts); feat[10]=np.sqrt(np.var(pts[:,0])+np.var(pts[:,1])); feat[11]=np.max(pts[:,2])-np.min(pts[:,2])
    t=frame_data.get('trackData',[])
    if len(t)>0:
        tr=t[0]
        for i,idx in enumerate(range(12,18)): feat[idx]=tr[i+1] if len(tr)>i+1 else 0.0
    h=frame_data.get('heightData',[])
    if len(h)>0: feat[18]=h[0][1] if len(h[0])>1 else 0.0; feat[19]=h[0][2] if len(h[0])>2 else 0.0
    return feat

def make_windows(features,label):
    w,l=[],[]
    for s in range(0,len(features)-WINDOW_SIZE+1,STRIDE): w.append(features[s:s+WINDOW_SIZE]); l.append(label)
    return w,l
print('Features defined.')

In [ ]:
def load_frames(path):
    with open(path,'r') as f: raw=json.load(f)
    if isinstance(raw,dict) and 'data' in raw: return raw['data']
    elif isinstance(raw,list): return raw
    elif isinstance(raw,dict) and 'frameData' in raw: return [raw]
    return []

def build_dataset(manifest_csv):
    manifest=pd.read_csv(manifest_csv)
    all_w,all_l,skipped=[],[],0
    mdir=Path(manifest_csv).parent
    for _,row in manifest.iterrows():
        label=int(row['label'])
        fp=Path(row['file'])
        if not fp.exists():
            sub='class_1_fall' if label==1 else 'class_0_nofall'
            fp=mdir/sub/fp.name
        if not fp.exists(): skipped+=1; continue
        try: frames=load_frames(fp)
        except: skipped+=1; continue
        if len(frames)==0: skipped+=1; continue
        feats,prev=[],None
        for frame in frames:
            fd=frame.get('frameData',frame)
            feat=extract_features(fd,prev); feats.append(feat); prev=feat
        feats=np.array(feats)
        if len(feats)<WINDOW_SIZE: skipped+=1; continue
        w,l=make_windows(feats,label); all_w.extend(w); all_l.extend(l)
    print(f'Skipped {skipped} files')
    return np.array(all_w,dtype=np.float32),np.array(all_l,dtype=np.int64)

In [ ]:
X,y=build_dataset(MANIFEST_CSV)
print(f'X={X.shape}, y={y.shape}')
nf=int((y==0).sum()); f=int((y==1).sum())
print(f'NO-FALL windows : {nf}'); print(f'FALL windows    : {f}'); print(f'Balance ratio   : {nf/f:.1f}x')

In [ ]:
X_tv,X_test,y_tv,y_test=train_test_split(X,y,test_size=0.15,stratify=y,random_state=42)
X_train,X_val,y_train,y_val=train_test_split(X_tv,y_tv,test_size=0.176,stratify=y_tv,random_state=42)
print(f'Train:{len(X_train)} Val:{len(X_val)} Test:{len(X_test)}')
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train.reshape(-1,20)).reshape(X_train.shape)
X_val_s=scaler.transform(X_val.reshape(-1,20)).reshape(X_val.shape)
X_test_s=scaler.transform(X_test.reshape(-1,20)).reshape(X_test.shape)
with open('fall_scaler.pkl','wb') as f: pickle.dump(scaler,f)
print('Scaler saved.')

In [ ]:
def make_loader(X,y,shuffle=True):
    ds=TensorDataset(torch.FloatTensor(X).to(device),torch.LongTensor(y).to(device))
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle)
train_loader=make_loader(X_train_s,y_train)
val_loader=make_loader(X_val_s,y_val,shuffle=False)
test_loader=make_loader(X_test_s,y_test,shuffle=False)
print(f'Loaders ready: {len(train_loader)}/{len(val_loader)}/{len(test_loader)}')

In [ ]:
class FallDetectionTransformerCNNLSTM(nn.Module):
    def __init__(self,input_size=20,d_model=64,nhead=4,num_transformer_layers=2,cnn_channels=32,lstm_hidden=64,dropout=0.4):
        super().__init__()
        self.input_proj=nn.Linear(input_size,d_model)
        enc=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=128,dropout=dropout,batch_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=num_transformer_layers)
        self.cnn=nn.Sequential(nn.Conv1d(d_model,cnn_channels,3,padding=1),nn.ReLU(),nn.BatchNorm1d(cnn_channels),nn.Dropout(dropout),nn.MaxPool1d(2))
        self.lstm=nn.LSTM(cnn_channels,lstm_hidden,1,batch_first=True)
        self.classifier=nn.Sequential(nn.Dropout(dropout),nn.Linear(lstm_hidden,32),nn.ReLU(),nn.Dropout(0.2),nn.Linear(32,2))
    def forward(self,x):
        x=self.input_proj(x); x=self.transformer(x); x=x.transpose(1,2); x=self.cnn(x); x=x.transpose(1,2)
        _,(h,_)=self.lstm(x); return self.classifier(h.squeeze(0))

model=FallDetectionTransformerCNNLSTM().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
cw=compute_class_weight('balanced',classes=np.array([0,1]),y=y_train)
criterion=nn.CrossEntropyLoss(weight=torch.FloatTensor(cw).to(device),label_smoothing=0.1)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS,eta_min=1e-5)
best_val_loss=float('inf'); patience_counter=0
history={'train_loss':[],'val_loss':[],'val_acc':[]}
for epoch in range(MAX_EPOCHS):
    model.train(); tl=0
    for xb,yb in train_loader:
        optimizer.zero_grad(); loss=criterion(model(xb),yb); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); optimizer.step(); tl+=loss.item()
    model.eval(); vl,correct,total=0,0,0
    with torch.no_grad():
        for xb,yb in val_loader:
            out=model(xb); vl+=criterion(out,yb).item(); correct+=(out.argmax(1)==yb).sum().item(); total+=len(yb)
    va=correct/total; tl/=len(train_loader); vl/=len(val_loader)
    history['train_loss'].append(tl); history['val_loss'].append(vl); history['val_acc'].append(va)
    scheduler.step()
    if vl<best_val_loss: best_val_loss=vl; patience_counter=0; torch.save(model.state_dict(),'fall_detection_model_best.pth')
    else:
        patience_counter+=1
        if patience_counter>=PATIENCE: print(f'Early stop @ epoch {epoch+1}'); break
    if (epoch+1)%10==0: print(f'Epoch {epoch+1:3d} | Train:{tl:.4f} Val:{vl:.4f} Acc:{va:.4f}')
print(f'Best val loss: {best_val_loss:.4f}')

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(history['train_loss'],label='Train'); plt.plot(history['val_loss'],label='Val')
plt.title('Loss'); plt.legend(); plt.grid(True,alpha=0.3)
plt.subplot(1,2,2); plt.plot(history['val_acc'],color='green'); plt.title('Val Accuracy'); plt.ylim(0,1); plt.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('training_curves.png',dpi=150); plt.show(); print('Saved training_curves.png')

In [ ]:
model.load_state_dict(torch.load('fall_detection_model_best.pth',map_location=device))
model.eval()
all_preds,all_true,all_probs=[],[],[]
with torch.no_grad():
    for xb,yb in test_loader:
        out=model(xb); probs=torch.softmax(out,dim=1)[:,1]
        all_preds.extend(out.argmax(1).cpu().numpy()); all_true.extend(yb.cpu().numpy()); all_probs.extend(probs.cpu().numpy())
print(classification_report(all_true,all_preds,target_names=['NO-FALL','FALL']))
fig,axes=plt.subplots(1,2,figsize=(12,4))
cm=confusion_matrix(all_true,all_preds)
sns.heatmap(cm,annot=True,fmt='d',ax=axes[0],cmap='Blues',xticklabels=['NO-FALL','FALL'],yticklabels=['NO-FALL','FALL']); axes[0].set_title('Confusion Matrix')
fpr,tpr,_=roc_curve(all_true,all_probs); auc=roc_auc_score(all_true,all_probs)
axes[1].plot(fpr,tpr,label=f'AUC={auc:.3f}'); axes[1].plot([0,1],[0,1],'--',color='gray')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('evaluation.png',dpi=150); plt.show(); print('Saved evaluation.png')

In [ ]:
for fn in ['fall_detection_model_best.pth','fall_scaler.pkl','evaluation.png','training_curves.png']:
    if os.path.exists(fn): print(f'{fn}: {os.path.getsize(fn)/1024:.1f} KB')
    else: print(f'{fn}: NOT FOUND')
print('Done! FALL when model(x).argmax()==1')